In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("GoldToWarehouse") \
    .config(
        "spark.hadoop.hive.metastore.uris",
        "thrift://docker-hive-hive-metastore-1:9083"
    ) \
    .enableHiveSupport() \
    .getOrCreate()

In [2]:
base_gold_path = "hdfs://docker-hive-namenode-1:8020/data-lake/gold"

gold_revenue_produk = spark.read.parquet(
    f"{base_gold_path}/revenue_produk"
)

gold_revenue_kota = spark.read.parquet(
    f"{base_gold_path}/revenue_kota"
)

gold_revenue_harian = spark.read.parquet(
    f"{base_gold_path}/revenue_harian"
)

In [3]:
gold_revenue_produk.show()
gold_revenue_kota.show()
gold_revenue_harian.show()

+------+----------+-------------+
|produk|  kategori|total_revenue|
+------+----------+-------------+
| Mouse|Elektronik|       450000|
|Pulpen|   Edukasi|        50000|
|  Buku|   Edukasi|       375000|
|Laptop|Elektronik|     24000000|
+------+----------+-------------+

+--------+-------------+
|    kota|total_revenue|
+--------+-------------+
|  Kediri|        50000|
| Jakarta|     24000000|
|Surabaya|       375000|
| Bandung|       450000|
+--------+-------------+

+----------+-------------+
|   tanggal|total_revenue|
+----------+-------------+
|2026-05-01|      8300000|
|2026-05-02|      8225000|
|2026-05-03|      8050000|
|2026-05-04|       300000|
+----------+-------------+



In [4]:
spark.sql("CREATE DATABASE IF NOT EXISTS warehouse")

DataFrame[]

In [5]:
gold_revenue_produk.write \
    .mode("overwrite") \
    .saveAsTable("warehouse.gold_revenue_produk")

gold_revenue_kota.write \
    .mode("overwrite") \
    .saveAsTable("warehouse.gold_revenue_kota")

gold_revenue_harian.write \
    .mode("overwrite") \
    .saveAsTable("warehouse.gold_revenue_harian")

In [6]:
spark.sql("SHOW TABLES IN warehouse").show(truncate=False)

+---------+-------------------+-----------+
|namespace|tableName          |isTemporary|
+---------+-------------------+-----------+
|warehouse|dim_produk         |false      |
|warehouse|fact_sales         |false      |
|warehouse|gold_revenue_harian|false      |
|warehouse|gold_revenue_kota  |false      |
|warehouse|gold_revenue_produk|false      |
|warehouse|revenue_kategori   |false      |
|warehouse|revenue_produk     |false      |
+---------+-------------------+-----------+



In [7]:
spark.sql("""
SELECT *
FROM warehouse.gold_revenue_produk
""").show()

+------+----------+-------------+
|produk|  kategori|total_revenue|
+------+----------+-------------+
| Mouse|Elektronik|       450000|
|Pulpen|   Edukasi|        50000|
|  Buku|   Edukasi|       375000|
|Laptop|Elektronik|     24000000|
+------+----------+-------------+

